In [11]:
import asyncio
import importlib.util
import os
from pathlib import Path

In [12]:
# Jupyter sets the current working directory to the folder containing the notebook.
# However, the original python script expects to run from the project root.
# We also make sure this behaves safely if the cell is run multiple times.
cwd = Path.cwd().resolve()
if cwd.name == "label" and cwd.parent.name == "scripts":
    os.chdir(cwd.parent.parent)
    print(f"Changed working directory to project root: {Path.cwd()}")
else:
    # Provide a fallback just in case it is already run from the project root
    print(f"Current working directory is already: {Path.cwd()}")

Current working directory is already: D:\Work\Research_Project\anaconda_research_project


In [13]:
# =========================================================
# Config
# =========================================================

# Existing script path (relative to the project root)
TARGET_SCRIPT = "trec_label_concurrent.py"

# Model settings
MODELS = ["qwen.qwen3-32b-v1:0"]
#MODELS = ["openai.gpt-oss-20b-1:0"]
MAX_TOKENS = 10000
TARGET_PATH = Path("scripts") / "label" / TARGET_SCRIPT

# Languages only
LANGUAGES = [
    "ru_instruct",
    "zh_instruct",
    "ga_instruct",
    "ar_instruct",
    "fr_instruct",
    "vi_instruct",
    "sw_instruct",
    "ga_instruct",
    "eng_instruct",
    "hi_instruct",
    "he_instruct",
]

# Shared part range for all languages
START_PART = 1
END_PART = 6

In [14]:
# =========================================================
# Load target script dynamically
# =========================================================

if not TARGET_PATH.exists():
    raise FileNotFoundError(f"Could not find target script: {TARGET_PATH.resolve()}")

spec = importlib.util.spec_from_file_location("label_runner_target", TARGET_PATH)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)


def set_run_globals(mod, lang: str, start_part: int, end_part: int) -> None:
    mod.LANG = lang
    mod.START_PART = start_part
    mod.END_PART = end_part
    mod.MODELS = MODELS
    if hasattr(mod, "INFERENCE_CONFIG"):
        mod.INFERENCE_CONFIG["maxTokens"] = MAX_TOKENS
    else:
        mod.INFERENCE_CONFIG = {"maxTokens": MAX_TOKENS, "temperature": 0.0, "topP": 1.0}

    if lang == "raw":
        mod.PART_DIR = Path(f"retrieved/trec_dl_{mod.TREC_DL_YEAR}/judged/")
    else:
        mod.PART_DIR = Path(f"retrieved/trec_dl_{mod.TREC_DL_YEAR}/{lang}/")

    mod.PART_PATTERN = f"all_topics_trecdl_{mod.TREC_DL_YEAR}_part{{n}}.csv"

In [15]:
async def run_all_languages():
    for lang in LANGUAGES:
        print("\n" + "=" * 80)
        print(
            f"[RUNNER] Starting language={lang} | "
            f"parts={START_PART}..{END_PART}"
        )
        print("=" * 80)

        set_run_globals(module, lang, START_PART, END_PART)

        try:
            await module.main()
            print(f"[RUNNER] Finished language={lang}")
        except KeyboardInterrupt:
            print(f"\n[RUNNER] Interrupted while processing language={lang}")
            break
        except Exception as e:
            print(f"[RUNNER] Error while processing language={lang}: {e}")

# Run the main function. Notice we await directly since IPython already runs an event loop.
await run_all_languages()


[RUNNER] Starting language=ru_instruct | parts=1..6
[STOP] Press 'Q' to stop gracefully.

--- Running inference for model: qwen.qwen3-32b-v1:0 (run_id=20260322_173553, LANG=ru_instruct, mode=replace) ---
[STOP] Press 'Q' at any time to stop after the current in-flight items].
[all_topics_trecdl_2022_part4.csv] Loaded 512 rows
[HEADER] LANG='ru_instruct' | output columns = ['qid', 'query', 'pid', 'passage', 'relevance', 'unique_string', 'unique_string_ru', 'passage_injected', 'llm_relevance']
[CONCURRENCY] row_workers=2 queue_max=4
[all_topics_trecdl_2022_part1.csv] Loaded 508 rows
[HEADER] LANG='ru_instruct' | output columns = ['qid', 'query', 'pid', 'passage', 'relevance', 'unique_string', 'unique_string_ru', 'passage_injected', 'llm_relevance']
[CONCURRENCY] row_workers=2 queue_max=4
[all_topics_trecdl_2022_part6.csv] Loaded 171 rows
[HEADER] LANG='ru_instruct' | output columns = ['qid', 'query', 'pid', 'passage', 'relevance', 'unique_string', 'unique_string_ru', 'passage_injected',